# Practice 48 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — DateOffset

`pd.DateOffset` adds calendar-aware amounts of time to dates. Unlike `pd.Timedelta` (which adds a fixed number of seconds), DateOffset respects month lengths, leap years, etc.

```python
date + pd.DateOffset(days=7)    # 7 calendar days later
date + pd.DateOffset(months=1)  # 1 month later (handles varying month lengths)
date + pd.DateOffset(years=1)   # 1 year later
```

Works element-wise on a Series too: `df['date'] + pd.DateOffset(months=3)`.

Use the project timeline below:

- Add a `deadline` column: each event's `start_date` + its `duration_weeks` weeks (use `DateOffset(weeks=...)`). You'll need to apply this row by row — try `.apply()` with `axis=1`.
- Add a `review_date` column: 2 weeks before each `deadline`.
- Which events have a `deadline` in Q2 (April–June)? Filter using `.dt.quarter`.
- How many days between the earliest `start_date` and the latest `deadline`? Use `.dt` arithmetic.

In [ ]:
projects = pd.DataFrame({
    'event':          ['Kickoff', 'Design', 'Development', 'Testing', 'Launch'],
    'start_date':     pd.to_datetime(['2024-01-08', '2024-01-22', '2024-02-05', '2024-04-01', '2024-05-13']),
    'duration_weeks': [2, 6, 10, 6, 2],
})

# Your code here

projects['deadline'] = projects.apply(lambda row: row['start_date']
+ pd.DateOffset(weeks=row['duration_weeks']), axis = 1)

projects['review_date'] = projects.apply(lambda row: row['deadline'] - pd.DateOffset(weeks=2), axis = 1)
projects['review_date'] = projects['deadline'] - pd.DateOffset(weeks=2)

projects[projects['deadline'].dt.quarter == 2]

e = projects['start_date'].min()
l = projects['deadline'].max()

(l-e).days

140

---
## Level 2 — freq-aware `shift()`

Two different things both called `shift()`:

```python
df.shift(1)              # shifts VALUES down 1 row — index unchanged, first row becomes NaN
df.shift(1, freq='MS')   # shifts the INDEX forward 1 month — values stay in place
```

The second form is useful for comparing a time series against a lagged version of itself (e.g. this month vs last month).

Common freq strings: `'D'` (day), `'MS'` (month start), `'ME'` (month end), `'QS'` (quarter start), `'YS'` (year start).

Use the monthly revenue data below:

- Add a `prev_month` column: revenue from the previous month. Use `shift(1)` (no freq).
- Add `mom_change`: month-over-month change in revenue (current minus previous).
- Add `mom_pct`: month-over-month percentage change. Round to 2 decimal places.
- Which month had the largest single-month increase? The largest drop?
- Use `shift(1, freq='MS')` to create a lagged copy of the DataFrame. What does its index look like compared to the original?

In [55]:
revenue = pd.DataFrame(
    {'revenue': [142000, 138000, 155000, 162000, 149000, 171000,
                 168000, 183000, 159000, 175000, 191000, 204000]},
    index=pd.date_range('2024-01', periods=12, freq='MS')
)

# Your code here
revenue['prev_month'] = revenue['revenue'].shift(1)
revenue['mom_change'] = revenue['revenue'] - revenue['prev_month']
revenue['mom_pct'] = round(revenue['mom_change']/revenue['prev_month'] ,2)
revenue['mom_change'].idxmax()
revenue['mom_change'].idxmin()

cp = revenue['revenue'].shift(1, freq='MS')

### index changed shifted to a month later all 

In [57]:
cp

2024-02-01    142000
2024-03-01    138000
2024-04-01    155000
2024-05-01    162000
2024-06-01    149000
2024-07-01    171000
2024-08-01    168000
2024-09-01    183000
2024-10-01    159000
2024-11-01    175000
2024-12-01    191000
2025-01-01    204000
Freq: MS, Name: revenue, dtype: int64

In [56]:
revenue

,revenue,prev_month,mom_change,mom_pct
2024-01-01,142000,NaN,NaN,NaN
2024-02-01,138000,142000.0,-4000.0,-0.03
2024-03-01,155000,138000.0,17000.0,0.12
2024-04-01,162000,155000.0,7000.0,0.05
2024-05-01,149000,162000.0,-13000.0,-0.08
2024-06-01,171000,149000.0,22000.0,0.15
2024-07-01,168000,171000.0,-3000.0,-0.02
2024-08-01,183000,168000.0,15000.0,0.09
2024-09-01,159000,183000.0,-24000.0,-0.13
2024-10-01,175000,159000.0,16000.0,0.10


In [54]:
revenue = pd.DataFrame(
    {'revenue': [142000, 138000, 155000, 162000, 149000, 171000,
                 168000, 183000, 159000, 175000, 191000, 204000]},
    index=pd.date_range('2024-01', periods=12, freq='MS')
)

revenue['prev_month'] = revenue['revenue'].shift(1)
revenue

,revenue,prev_month
2024-01-01,142000,NaN
2024-02-01,138000,142000.0
2024-03-01,155000,138000.0
2024-04-01,162000,155000.0
2024-05-01,149000,162000.0
2024-06-01,171000,149000.0
2024-07-01,168000,171000.0
2024-08-01,183000,168000.0
2024-09-01,159000,183000.0
2024-10-01,175000,159000.0


---
## Level 3 — timezones

Two steps for working with timezones:

```python
# Step 1 — attach a timezone to naive datetimes (no existing tz info)
s.dt.tz_localize('UTC')

# Step 2 — convert to a different timezone (series must already be tz-aware)
s.dt.tz_convert('US/Eastern')
s.dt.tz_convert('Europe/London')
s.dt.tz_convert('Asia/Tokyo')
```

After converting, `.dt.hour` gives the local hour in the new timezone.

Use the server log data below (all timestamps are UTC):

- Localize `timestamp` to UTC, then add two columns: `time_eastern` and `time_tokyo`.
- Add `hour_eastern` and `hour_tokyo`: the local hour in each timezone (use `.dt.hour`).
- Flag `is_business_eastern`: True where `hour_eastern` is between 9 and 17 (inclusive).
- Among `ERROR` events, what fraction occurred during Eastern business hours?
- Use `.pipe()` to structure the above as `localize(df)` and `classify(df)` functions.

In [42]:
logs = pd.DataFrame({
    'timestamp': pd.to_datetime([
        '2024-03-15 02:00', '2024-03-15 09:30', '2024-03-15 13:00',
        '2024-03-15 17:45', '2024-03-15 21:00', '2024-03-16 04:00',
        '2024-03-16 11:00', '2024-03-16 15:30', '2024-03-16 23:00',
    ]),
    'level':  ['INFO', 'ERROR', 'INFO', 'ERROR', 'INFO', 'ERROR', 'INFO', 'ERROR', 'INFO'],
    'region': ['APAC', 'AMER', 'EMEA', 'AMER', 'APAC', 'EMEA', 'AMER', 'APAC', 'EMEA'],
})

# Your code here
logs['timestamp'] = logs['timestamp'].dt.tz_localize('UTC')
logs['time_eastern'] = logs['timestamp'].dt.tz_convert('US/Eastern')
logs['time_tokyo'] = logs['timestamp'].dt.tz_convert('Asia/Tokyo')

logs['hour_eastern'] =logs['time_eastern'].dt.hour
logs['hour_tokyo'] = logs['time_tokyo'].dt.hour

logs['is_business_eastern'] = (logs['hour_eastern']>=9) & (logs['hour_eastern']<=17)

logs.loc[logs['level'] == 'ERROR','is_business_eastern'].mean()

def localize(logs):
    logs['timestamp'] = logs['timestamp'].dt.tz_localize('UTC')
    logs['time_eastern'] = logs['timestamp'].dt.tz_convert('US/Eastern')
    logs['time_tokyo'] = logs['timestamp'].dt.tz_convert('Asia/Tokyo')
    logs['hour_eastern'] =logs['time_eastern'].dt.hour
    logs['hour_tokyo'] = logs['time_tokyo'].dt.hour
    return logs 

def classify(logs):
    logs['is_business_eastern'] = (logs['hour_eastern']>=9) & (logs['hour_eastern']<=17)
    return logs 


In [44]:
logs = pd.DataFrame({
    'timestamp': pd.to_datetime([
        '2024-03-15 02:00', '2024-03-15 09:30', '2024-03-15 13:00',
        '2024-03-15 17:45', '2024-03-15 21:00', '2024-03-16 04:00',
        '2024-03-16 11:00', '2024-03-16 15:30', '2024-03-16 23:00',
    ]),
    'level':  ['INFO', 'ERROR', 'INFO', 'ERROR', 'INFO', 'ERROR', 'INFO', 'ERROR', 'INFO'],
    'region': ['APAC', 'AMER', 'EMEA', 'AMER', 'APAC', 'EMEA', 'AMER', 'APAC', 'EMEA'],
})
logs.copy().pipe(localize).pipe(classify)


,timestamp,level,region,time_eastern,time_tokyo,hour_eastern,hour_tokyo,is_business_eastern
0,2024-03-15 02:00:00+00:00,INFO,APAC,2024-03-14 22:00:00-04:00,2024-03-15 11:00:00+09:00,22,11,False
1,2024-03-15 09:30:00+00:00,ERROR,AMER,2024-03-15 05:30:00-04:00,2024-03-15 18:30:00+09:00,5,18,False
2,2024-03-15 13:00:00+00:00,INFO,EMEA,2024-03-15 09:00:00-04:00,2024-03-15 22:00:00+09:00,9,22,True
3,2024-03-15 17:45:00+00:00,ERROR,AMER,2024-03-15 13:45:00-04:00,2024-03-16 02:45:00+09:00,13,2,True
4,2024-03-15 21:00:00+00:00,INFO,APAC,2024-03-15 17:00:00-04:00,2024-03-16 06:00:00+09:00,17,6,True
5,2024-03-16 04:00:00+00:00,ERROR,EMEA,2024-03-16 00:00:00-04:00,2024-03-16 13:00:00+09:00,0,13,False
6,2024-03-16 11:00:00+00:00,INFO,AMER,2024-03-16 07:00:00-04:00,2024-03-16 20:00:00+09:00,7,20,False
7,2024-03-16 15:30:00+00:00,ERROR,APAC,2024-03-16 11:30:00-04:00,2024-03-17 00:30:00+09:00,11,0,True
8,2024-03-16 23:00:00+00:00,INFO,EMEA,2024-03-16 19:00:00-04:00,2024-03-17 08:00:00+09:00,19,8,False
